# 10 - Final İç Test

Bu notebook, Stage 09'da sabitlenen detection parametreleriyle Dataset1 ve Dataset2 test splitleri üzerinde iç final değerlendirme yapar.

Amaç model veya parametre seçmek değil; sabit aday modellerin test performansını raporlamak ve Stage 11 öncesi iç test sonuçlarını üretmektir.


## 1. Kütüphaneler

Bu bölümde tablo işlemleri, görüntü işleme, model yükleme, feature çıkarma ve metrik hesaplama için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import math
import time
import traceback
import warnings

import numpy as np
import pandas as pd

import joblib
import cv2
import matplotlib.pyplot as plt

from skimage.feature import hog, local_binary_pattern

warnings.filterwarnings("default")
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but StandardScaler was fitted with feature names",
    category=UserWarning,
)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)

print("Kütüphaneler yüklendi.")


## 2. Ayarlar

Bu bölümde çalıştırılacak test modları, overwrite davranışı ve detection metrik eşikleri tanımlanır.


In [ ]:
RANDOM_STATE = 42

# User-approved Stage 10 execution flags.
RUN_NATURAL_700 = True
RUN_CONTROLLED_550 = True
RUN_FULL_TEST = True
COMPUTE_AP_MAP = True

# Runtime / output behavior.
OVERWRITE_PARTIAL_RESULTS = False
OVERWRITE_TABLES = False
MAX_CANDIDATES_PER_IMAGE = 500
FEATURE_EXTRACTION_BATCH_SIZE = 512
DISPLAY_PREVIEW_ROWS = 10

# Stage 10 metric settings.
MATCHING_IOU_THRESHOLDS = [0.20, 0.30]
AP_REPORT_IOU_THRESHOLDS = [0.20, 0.30, 0.50]
MAP_50_95_THRESHOLDS = [round(x, 2) for x in np.arange(0.50, 0.96, 0.05)]
DEFAULT_PRIMARY_METRIC = "detection_f1_iou_0_20"
STRICT_COMPARISON_METRIC = "detection_f1_iou_0_30"
POSTPROCESS_METHOD = "weighted_center_clustering"

NATURAL_SUBSET_TARGET_PER_DATASET = 700
CONTROLLED_SUBSET_TARGETS = {
    "zero": 280,
    "one": 220,
    "two_or_more": 50,
}

EXPECTED_TEST_DISTRIBUTIONS = {
    "dataset1_varroa": {"total": 3408, "zero": 2190, "one": 998, "two": 210, "three_plus": 10},
    "dataset2_honeybee": {"total": 790, "zero": 286, "one": 443, "two": 54, "three_plus": 7},
}

print("Ayarlar yüklendi.")


## 3. Dosya Yolları

Bu bölümde proje kökü, Stage 10 çıktı klasörü, Stage 09 audit tabloları, ham test verisi ve model havuzu yolları tanımlanır.


In [ ]:
def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    start_path = Path(start_path).resolve()
    for candidate in [start_path] + list(start_path.parents):
        has_repo_layout = (candidate / "approaches" / "traditional_ml").exists()
        has_data_dir = (candidate / "data").exists()
        if has_repo_layout and has_data_dir:
            return candidate
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Proje kökü bulunamadı. Notebook'u repo içinde çalıştırın.")

APPROACH_ROOT = PROJECT_ROOT / "approaches" / "traditional_ml"
STAGE_NAME = "10_final_internal_test"
NOTEBOOK_NAME = "01_internal_detection_test"

OUTPUT_DIR = APPROACH_ROOT / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
PARTIAL_DIR = TABLES_DIR / "partial_model_results"
INPUT_DIR = APPROACH_ROOT / "inputs" / STAGE_NAME

for folder in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR, PARTIAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_FEATURES_DIR = PROJECT_ROOT / "data" / "features"
SELECTED_MODEL_POOL_DIR = PROJECT_ROOT / "outputs" / "models" / "selected_classification_model_pool"

STAGE09_AUDIT_CANDIDATES = [
    INPUT_DIR / "threshold_refinement_large_subset__ranked_detection_tuning_results_by_iou_0_2.csv",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SELECTED_MODEL_POOL_DIR:", SELECTED_MODEL_POOL_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("STAGE09_AUDIT_CANDIDATES:", STAGE09_AUDIT_CANDIDATES)


## 4. Kayıt Yardımcıları

Bu bölümde CSV çıktıları ve kısa kayıt listesi için ortak yardımcı fonksiyonlar tanımlanır.


In [ ]:
SAVED_FILES = []
NOTEBOOK_START_TIME = datetime.now()


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def record_saved_file(path, description):
    path = Path(path)
    SAVED_FILES.append({
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "path": relative_path(path),
        "description": description,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    })


def save_csv(df, path, description, index=False, overwrite=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite is None:
        overwrite = OVERWRITE_TABLES

    if path.exists() and not overwrite:
        print(f"[SKIP] Mevcut CSV korundu: {relative_path(path)}")
        try:
            return pd.read_csv(path)
        except pd.errors.EmptyDataError:
            print(f"[INFO] Mevcut CSV boş olduğu için güncel tablo bellekte kullanılacak: {relative_path(path)}")
            return df

    df.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"[SAVE] CSV kaydedildi: {relative_path(path)} | shape={df.shape}")
    record_saved_file(path, description)
    return df


def safe_divide(numerator, denominator):
    return float(numerator) / float(denominator) if denominator else 0.0


def metric_key(iou_threshold):
    return str(float(iou_threshold)).replace(".", "_")


def model_stem(model_file_name):
    return Path(str(model_file_name)).stem


print("Yardımcı fonksiyonlar hazır.")


## 5. Veri Seti Ayarları

Bu bölümde Dataset1 ve Dataset2 test görüntüleri, test etiketleri ve PCA artifact klasörleri tanımlanır.


In [ ]:
DATASET_CONFIGS = {
    "dataset1_varroa": {
        "dataset_short": "dataset1",
        "test_images_dir": DATA_RAW_DIR / "dataset1_varroa" / "test" / "images",
        "test_labels_dir": DATA_RAW_DIR / "dataset1_varroa" / "test" / "labels",
        "pca_artifacts_dir": DATA_FEATURES_DIR / "dataset1_varroa" / "pca_artifacts",
        "positive_class_ids": {1},
    },
    "dataset2_honeybee": {
        "dataset_short": "dataset2",
        "test_images_dir": DATA_RAW_DIR / "dataset2_honeybee" / "test" / "images",
        "test_labels_dir": DATA_RAW_DIR / "dataset2_honeybee" / "test" / "labels",
        "pca_artifacts_dir": DATA_FEATURES_DIR / "dataset2_honeybee" / "pca_artifacts",
        "positive_class_ids": {0},
    },
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for dataset_name, cfg in DATASET_CONFIGS.items():
    print(dataset_name)
    print("  test_images_dir:", cfg["test_images_dir"], "exists=", cfg["test_images_dir"].exists())
    print("  test_labels_dir:", cfg["test_labels_dir"], "exists=", cfg["test_labels_dir"].exists())
    print("  pca_artifacts_dir:", cfg["pca_artifacts_dir"], "exists=", cfg["pca_artifacts_dir"].exists())


## 6. Sabit Model ve Parametre Havuzu

Bu bölümde Stage 10'da değerlendirilecek sabit model listesi ve detection parametreleri tanımlanır.


In [ ]:
STAGE10_FIXED_PARAMS = pd.DataFrame([
    {
        "dataset_name": "dataset1_varroa",
        "algorithm_name": "rbf_svm",
        "model_file_name": "d1_rbfsvm_006__centered80_neg4x__hog16_hsv_lbp.joblib",
        "model_id": "d1_rbfsvm_006",
        "patch_set": "centered80_neg4x",
        "patch_size": 80,
        "feature_set": "hog16_hsv_lbp",
        "threshold_column": "decision_function_score_threshold",
        "threshold_value": 1.20,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.25,
        "weighted_box_size_factor": 1.00,
    },
    {
        "dataset_name": "dataset1_varroa",
        "algorithm_name": "rbf_svm",
        "model_file_name": "d1_rbfsvm_007__centered80_neg4x__hog16_pca_hsv_lbp.joblib",
        "model_id": "d1_rbfsvm_007",
        "patch_set": "centered80_neg4x",
        "patch_size": 80,
        "feature_set": "hog16_pca_hsv_lbp",
        "threshold_column": "decision_function_score_threshold",
        "threshold_value": 1.40,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.25,
        "weighted_box_size_factor": 1.25,
    },
    {
        "dataset_name": "dataset1_varroa",
        "algorithm_name": "linear_svm",
        "model_file_name": "d1_linsvm_024__centered80_neg4x__hog16_pca_hsv_lbp.joblib",
        "model_id": "d1_linsvm_024",
        "patch_set": "centered80_neg4x",
        "patch_size": 80,
        "feature_set": "hog16_pca_hsv_lbp",
        "threshold_column": "decision_function_score_threshold",
        "threshold_value": 1.60,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.25,
        "weighted_box_size_factor": 1.00,
    },
    {
        "dataset_name": "dataset1_varroa",
        "algorithm_name": "linear_svm",
        "model_file_name": "d1_linsvm_004__centered80_balanced__hog16_hsv_lbp.joblib",
        "model_id": "d1_linsvm_004",
        "patch_set": "centered80_balanced",
        "patch_size": 80,
        "feature_set": "hog16_hsv_lbp",
        "threshold_column": "decision_function_score_threshold",
        "threshold_value": 2.20,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.20,
        "weighted_box_size_factor": 1.00,
    },
    {
        "dataset_name": "dataset1_varroa",
        "algorithm_name": "linear_svm",
        "model_file_name": "d1_linsvm_011__centered80_balanced__hog16_pca_hsv_lbp.joblib",
        "model_id": "d1_linsvm_011",
        "patch_set": "centered80_balanced",
        "patch_size": 80,
        "feature_set": "hog16_pca_hsv_lbp",
        "threshold_column": "decision_function_score_threshold",
        "threshold_value": 2.20,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.20,
        "weighted_box_size_factor": 1.00,
    },
    {
        "dataset_name": "dataset2_honeybee",
        "algorithm_name": "random_forest",
        "model_file_name": "d2_rf_006__centered48_neg4x__hog16_hsv_lbp.joblib",
        "model_id": "d2_rf_006",
        "patch_set": "centered48_neg4x",
        "patch_size": 48,
        "feature_set": "hog16_hsv_lbp",
        "threshold_column": "predict_proba_score_threshold",
        "threshold_value": 0.60,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.20,
        "weighted_box_size_factor": 1.00,
    },
    {
        "dataset_name": "dataset2_honeybee",
        "algorithm_name": "random_forest",
        "model_file_name": "d2_rf_007__centered48_neg4x__hog16_pca_hsv_lbp.joblib",
        "model_id": "d2_rf_007",
        "patch_set": "centered48_neg4x",
        "patch_size": 48,
        "feature_set": "hog16_pca_hsv_lbp",
        "threshold_column": "predict_proba_score_threshold",
        "threshold_value": 0.70,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.20,
        "weighted_box_size_factor": 1.00,
    },
    {
        "dataset_name": "dataset2_honeybee",
        "algorithm_name": "random_forest",
        "model_file_name": "d2_rf_003__centered48_balanced__hog16_pca_hsv_lbp.joblib",
        "model_id": "d2_rf_003",
        "patch_set": "centered48_balanced",
        "patch_size": 48,
        "feature_set": "hog16_pca_hsv_lbp",
        "threshold_column": "predict_proba_score_threshold",
        "threshold_value": 0.85,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.20,
        "weighted_box_size_factor": 1.00,
    },
    {
        "dataset_name": "dataset2_honeybee",
        "algorithm_name": "rbf_svm",
        "model_file_name": "d2_rbfsvm_007__centered48_neg4x__hog16_pca_hsv_lbp.joblib",
        "model_id": "d2_rbfsvm_007",
        "patch_set": "centered48_neg4x",
        "patch_size": 48,
        "feature_set": "hog16_pca_hsv_lbp",
        "threshold_column": "decision_function_score_threshold",
        "threshold_value": 0.90,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.20,
        "weighted_box_size_factor": 1.00,
    },
    {
        "dataset_name": "dataset2_honeybee",
        "algorithm_name": "rbf_svm",
        "model_file_name": "d2_rbfsvm_006__centered48_neg4x__hog16_hsv_lbp.joblib",
        "model_id": "d2_rbfsvm_006",
        "patch_set": "centered48_neg4x",
        "patch_size": 48,
        "feature_set": "hog16_hsv_lbp",
        "threshold_column": "decision_function_score_threshold",
        "threshold_value": 0.80,
        "stride_divisor": 4,
        "cluster_iou_threshold": 0.20,
        "weighted_box_size_factor": 1.00,
    },
])

ALGORITHM_FOLDER_MAP = {
    "linear_svm": "linear_svm",
    "rbf_svm": "rbf_svm",
    "random_forest": "random_forest",
}

def resolve_model_path(row):
    model_file = str(row["model_file_name"])
    candidates = [
        SELECTED_MODEL_POOL_DIR / row["dataset_name"] / ALGORITHM_FOLDER_MAP[row["algorithm_name"]] / model_file,
        SELECTED_MODEL_POOL_DIR / row["dataset_name"] / row["algorithm_name"] / model_file,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

STAGE10_FIXED_PARAMS["model_path_resolved"] = STAGE10_FIXED_PARAMS.apply(resolve_model_path, axis=1).astype(str)
STAGE10_FIXED_PARAMS["model_exists"] = STAGE10_FIXED_PARAMS["model_path_resolved"].map(lambda p: Path(p).exists())
STAGE10_FIXED_PARAMS["postprocess_method"] = POSTPROCESS_METHOD

save_csv(STAGE10_FIXED_PARAMS, TABLES_DIR / "stage10_fixed_detection_parameters.csv", "Fixed Stage 10 candidate pool and detection parameters")

display(STAGE10_FIXED_PARAMS)
if not STAGE10_FIXED_PARAMS["model_exists"].all():
    missing = STAGE10_FIXED_PARAMS.loc[~STAGE10_FIXED_PARAMS["model_exists"], ["model_file_name", "model_path_resolved"]]
    warnings.warn("Some model artifacts were not found. Fix model paths before running evaluation.\n" + missing.to_string(index=False))


## 7. Stage 09 Parametre Kontrolü

Bu bölümde gömülü Stage 10 parametreleri, varsa Stage 09 tuning sonuçlarıyla karşılaştırılır.


In [ ]:
def find_stage09_audit_table():
    for path in STAGE09_AUDIT_CANDIDATES:
        if path.exists():
            return path
    return None

def audit_fixed_parameters_against_stage09():
    audit_path = find_stage09_audit_table()
    if audit_path is None:
        warnings.warn("No Stage 09 threshold-refinement CSV found. Continuing with embedded fixed Stage 10 parameter table.")
        return pd.DataFrame([{"audit_status": "SKIPPED", "message": "Stage 09 CSV not found"}])

    stage09_df = pd.read_csv(audit_path)
    compare_rows = []

    # Normalize likely column names from Stage 09 outputs.
    stage09 = stage09_df.copy()
    if "model_file_name" not in stage09.columns:
        return pd.DataFrame([{"audit_status": "SKIPPED", "message": f"No model_file_name in {audit_path}"}])

    for _, fixed in STAGE10_FIXED_PARAMS.iterrows():
        subset = stage09[stage09["model_file_name"].astype(str) == str(fixed["model_file_name"])]
        if subset.empty:
            compare_rows.append({
                "audit_status": "WARN",
                "model_file_name": fixed["model_file_name"],
                "message": "Model not found in Stage 09 audit CSV",
                "stage09_source": relative_path(audit_path),
            })
            continue

        # If ranked file has many rows, use first row as the best-ranked row for that model.
        row = subset.iloc[0]
        for col in ["threshold_value", "stride_divisor", "cluster_iou_threshold", "weighted_box_size_factor"]:
            if col not in row.index:
                compare_rows.append({
                    "audit_status": "WARN",
                    "model_file_name": fixed["model_file_name"],
                    "parameter": col,
                    "fixed_value": fixed[col],
                    "stage09_value": np.nan,
                    "matches": False,
                    "message": f"Column {col} missing in Stage 09 CSV",
                    "stage09_source": relative_path(audit_path),
                })
                continue
            fixed_val = float(fixed[col])
            stage09_val = float(row[col])
            compare_rows.append({
                "audit_status": "PASS" if np.isclose(fixed_val, stage09_val) else "WARN",
                "model_file_name": fixed["model_file_name"],
                "parameter": col,
                "fixed_value": fixed_val,
                "stage09_value": stage09_val,
                "matches": bool(np.isclose(fixed_val, stage09_val)),
                "message": "OK" if np.isclose(fixed_val, stage09_val) else "Fixed value differs from Stage 09 CSV row",
                "stage09_source": relative_path(audit_path),
            })

    return pd.DataFrame(compare_rows)

stage09_parameter_audit_df = audit_fixed_parameters_against_stage09()
save_csv(stage09_parameter_audit_df, TABLES_DIR / "stage09_parameter_audit.csv", "Audit of embedded fixed Stage 10 parameters against Stage 09 CSV when available")
display(stage09_parameter_audit_df.head(50))


## 8. Test Görüntü ve Ground Truth Yardımcıları

Bu bölümde test split görüntüleri ve YOLO etiketlerinden ground-truth kutuları hazırlanır.


In [ ]:
def find_image_files(images_dir):
    images_dir = Path(images_dir)
    return sorted([p for p in images_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])

def parse_yolo_label_file(label_path, image_width, image_height, positive_class_ids):
    boxes = []
    invalid_rows = 0
    label_path = Path(label_path)
    if not label_path.exists():
        return boxes, invalid_rows

    for row_index, line in enumerate(label_path.read_text(encoding="utf-8", errors="ignore").splitlines()):
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            invalid_rows += 1
            continue
        try:
            class_id = int(float(parts[0]))
            x_center, y_center, width, height = map(float, parts[1:5])
        except Exception:
            invalid_rows += 1
            continue

        if class_id not in positive_class_ids:
            continue
        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and width > 0 and height > 0):
            invalid_rows += 1
            continue

        x1 = max(0.0, min(float(image_width), (x_center - width / 2) * image_width))
        y1 = max(0.0, min(float(image_height), (y_center - height / 2) * image_height))
        x2 = max(0.0, min(float(image_width), (x_center + width / 2) * image_width))
        y2 = max(0.0, min(float(image_height), (y_center + height / 2) * image_height))
        if x2 <= x1 or y2 <= y1:
            invalid_rows += 1
            continue

        boxes.append({"class_id": class_id, "row_index": row_index, "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    return boxes, invalid_rows

def build_test_image_records(dataset_name):
    cfg = DATASET_CONFIGS[dataset_name]
    records = []
    for image_path in find_image_files(cfg["test_images_dir"]):
        image = cv2.imread(str(image_path))
        label_path = cfg["test_labels_dir"] / f"{image_path.stem}.txt"
        if image is None:
            records.append({
                "dataset_name": dataset_name,
                "split": "test",
                "image_path": relative_path(image_path),
                "label_path": relative_path(label_path),
                "image_id": image_path.stem,
                "image_file_name": image_path.name,
                "image_width": np.nan,
                "image_height": np.nan,
                "gt_boxes": [],
                "gt_count": 0,
                "invalid_yolo_row_count": np.nan,
                "readable": False,
            })
            continue
        h, w = image.shape[:2]
        gt_boxes, invalid_rows = parse_yolo_label_file(label_path, w, h, cfg["positive_class_ids"])
        records.append({
            "dataset_name": dataset_name,
            "split": "test",
            "image_path": relative_path(image_path),
            "label_path": relative_path(label_path),
            "image_id": image_path.stem,
            "image_file_name": image_path.name,
            "image_width": w,
            "image_height": h,
            "gt_boxes": gt_boxes,
            "gt_count": len(gt_boxes),
            "invalid_yolo_row_count": invalid_rows,
            "readable": True,
        })
    return pd.DataFrame(records)

def add_gt_count_group(df):
    df = df.copy()
    def group_count(value):
        value = int(value)
        if value == 0:
            return "zero"
        if value == 1:
            return "one"
        if value == 2:
            return "two"
        return "three_plus"
    df["gt_count_group"] = df["gt_count"].apply(group_count)
    return df

def add_controlled_count_group(df):
    df = df.copy()
    def group_count(value):
        value = int(value)
        if value == 0:
            return "zero"
        if value == 1:
            return "one"
        return "two_or_more"
    df["controlled_gt_count_group"] = df["gt_count"].apply(group_count)
    return df

test_records_by_dataset = {dataset: build_test_image_records(dataset) for dataset in DATASET_CONFIGS}

inventory_rows = []
for dataset_name, df in test_records_by_dataset.items():
    readable = df[df["readable"] == True].copy()
    grouped = add_gt_count_group(readable)["gt_count_group"].value_counts().to_dict()
    row = {
        "dataset_name": dataset_name,
        "test_image_count": int(len(df)),
        "readable_image_count": int(len(readable)),
        "gt_object_count": int(readable["gt_count"].sum()) if not readable.empty else 0,
        "zero": int(grouped.get("zero", 0)),
        "one": int(grouped.get("one", 0)),
        "two": int(grouped.get("two", 0)),
        "three_plus": int(grouped.get("three_plus", 0)),
        "positive_image_count": int((readable["gt_count"] > 0).sum()) if not readable.empty else 0,
        "invalid_yolo_row_count": int(pd.to_numeric(readable["invalid_yolo_row_count"], errors="coerce").fillna(0).sum()) if not readable.empty else 0,
    }
    inventory_rows.append(row)

    expected = EXPECTED_TEST_DISTRIBUTIONS.get(dataset_name)
    if expected:
        mismatches = []
        if row["readable_image_count"] != expected["total"]:
            mismatches.append(f"total expected={expected['total']} observed={row['readable_image_count']}")
        for key in ["zero", "one", "two", "three_plus"]:
            if row[key] != expected[key]:
                mismatches.append(f"{key} expected={expected[key]} observed={row[key]}")
        if mismatches:
            warnings.warn(f"Distribution mismatch for {dataset_name}: " + "; ".join(mismatches))

input_inventory_df = pd.DataFrame(inventory_rows)
save_csv(input_inventory_df, TABLES_DIR / "stage10_test_input_inventory.csv", "Stage 10 test split input inventory")
display(input_inventory_df)


## 9. Alt Küme Seçim Yardımcıları

Bu bölümde doğal 700 ve kontrollü 550 test alt kümeleri için seçim fonksiyonları tanımlanır.


In [ ]:
def select_proportional_stratified_subset(df, target_count, random_state):
    df = add_gt_count_group(df[df["readable"] == True].copy())

    if len(df) <= target_count:
        return df.sort_values("image_id").reset_index(drop=True)

    group_order = ["zero", "one", "two", "three_plus"]
    group_counts = df["gt_count_group"].value_counts().to_dict()

    allocations = {}
    fractional_parts = {}
    for group in group_order:
        available = int(group_counts.get(group, 0))
        exact = target_count * available / len(df)
        base = int(np.floor(exact))
        base = min(base, available)
        allocations[group] = base
        fractional_parts[group] = exact - base

    remaining_slots = target_count - sum(allocations.values())
    while remaining_slots > 0:
        candidates = [group for group in group_order if allocations[group] < int(group_counts.get(group, 0))]
        if not candidates:
            break
        best_group = max(candidates, key=lambda g: fractional_parts[g])
        allocations[best_group] += 1
        fractional_parts[best_group] = 0.0
        remaining_slots -= 1

    selected_parts = []
    for group in group_order:
        group_df = df[df["gt_count_group"] == group]
        n_group = int(allocations.get(group, 0))
        if n_group <= 0 or group_df.empty:
            continue
        selected_parts.append(group_df.sample(n=n_group, random_state=random_state))

    selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame(columns=df.columns)

    if len(selected) < target_count:
        selected_ids = set(selected["image_id"].astype(str)) if not selected.empty else set()
        remaining_df = df[~df["image_id"].astype(str).isin(selected_ids)]
        need = target_count - len(selected)
        if not remaining_df.empty:
            selected = pd.concat([
                selected,
                remaining_df.sample(n=min(need, len(remaining_df)), random_state=random_state),
            ], ignore_index=True)

    if len(selected) > target_count:
        selected = selected.sample(n=target_count, random_state=random_state)

    return selected.sort_values("image_id").reset_index(drop=True)

def select_controlled_subset(df, targets, random_state):
    df = add_controlled_count_group(df[df["readable"] == True].copy())
    selected_parts = []
    availability_rows = []

    for group, target_count in targets.items():
        group_df = df[df["controlled_gt_count_group"] == group]
        available = len(group_df)
        n_select = min(int(target_count), int(available))
        availability_rows.append({
            "controlled_gt_count_group": group,
            "available_count": int(available),
            "target_count": int(target_count),
            "selected_count": int(n_select),
            "target_met": bool(available >= target_count),
        })
        if available < target_count:
            warnings.warn(
                f"Controlled subset target not fully met for group={group}: "
                f"available={available}, target={target_count}. Selecting all available records."
            )
        if n_select > 0:
            selected_parts.append(group_df.sample(n=n_select, random_state=random_state))

    selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame(columns=df.columns)
    return selected.sort_values("image_id").reset_index(drop=True), pd.DataFrame(availability_rows)

def save_selected_images(df, path, description):
    selected_df = df.drop(columns=["gt_boxes"], errors="ignore").copy()

    for path_column in ["image_path", "label_path"]:
        if path_column in selected_df.columns:
            selected_df[path_column] = selected_df[path_column].map(relative_path)

    save_csv(selected_df, path, description)
    return selected_df

eval_modes = []
subset_inventory_rows = []

if RUN_NATURAL_700:
    for dataset_name, full_df in test_records_by_dataset.items():
        selected = select_proportional_stratified_subset(full_df, NATURAL_SUBSET_TARGET_PER_DATASET, RANDOM_STATE)
        eval_modes.append({"eval_mode": "natural_700", "dataset_name": dataset_name, "records": selected})
        save_selected_images(
            selected,
            TABLES_DIR / f"natural_700_selected_images_{dataset_name}.csv",
            f"Natural-distribution 700-image selected test subset for {dataset_name}",
        )

if RUN_CONTROLLED_550:
    availability_all = []
    for dataset_name, full_df in test_records_by_dataset.items():
        selected, availability_df = select_controlled_subset(full_df, CONTROLLED_SUBSET_TARGETS, RANDOM_STATE)
        availability_df["dataset_name"] = dataset_name
        availability_all.append(availability_df)
        eval_modes.append({"eval_mode": "controlled_550", "dataset_name": dataset_name, "records": selected})
        save_selected_images(
            selected,
            TABLES_DIR / f"controlled_550_selected_images_{dataset_name}.csv",
            f"Controlled 550-image selected test subset for {dataset_name}",
        )
    save_csv(pd.concat(availability_all, ignore_index=True), TABLES_DIR / "controlled_550_subset_availability.csv", "Controlled subset target availability")

if RUN_FULL_TEST:
    for dataset_name, full_df in test_records_by_dataset.items():
        selected = full_df[full_df["readable"] == True].copy().reset_index(drop=True)
        eval_modes.append({"eval_mode": "full_test", "dataset_name": dataset_name, "records": selected})

for mode in eval_modes:
    df = mode["records"]
    grouped = add_gt_count_group(df)["gt_count_group"].value_counts().to_dict() if not df.empty else {}
    controlled_grouped = add_controlled_count_group(df)["controlled_gt_count_group"].value_counts().to_dict() if not df.empty else {}
    subset_inventory_rows.append({
        "eval_mode": mode["eval_mode"],
        "dataset_name": mode["dataset_name"],
        "image_count": int(len(df)),
        "gt_object_count": int(df["gt_count"].sum()) if not df.empty else 0,
        "zero": int(grouped.get("zero", 0)),
        "one": int(grouped.get("one", 0)),
        "two": int(grouped.get("two", 0)),
        "three_plus": int(grouped.get("three_plus", 0)),
        "two_or_more": int(controlled_grouped.get("two_or_more", 0)),
    })

stage10_subset_inventory_df = pd.DataFrame(subset_inventory_rows)
save_csv(stage10_subset_inventory_df, TABLES_DIR / "stage10_subset_inventory.csv", "Stage 10 evaluation subset inventory")
display(stage10_subset_inventory_df)


## 10. Feature Çıkarma Yardımcıları

Bu bölümde HOG, HSV, LBP ve PCA feature çıkarma fonksiyonları tanımlanır.


In [ ]:
PCA_ARTIFACT_CACHE = {}

def parse_hog_variant(feature_set):
    if "hog8" in feature_set:
        return "hog8", 8
    if "hog16" in feature_set:
        return "hog16", 16
    return None, None

def is_pca_feature_set(feature_set):
    return "_pca" in feature_set

def uses_hog(feature_set):
    return "hog8" in feature_set or "hog16" in feature_set

def uses_hsv(feature_set):
    return "hsv" in feature_set

def uses_lbp(feature_set):
    return "lbp" in feature_set

def unwrap_model_artifact(artifact, artifact_path=None):
    """Return the estimator/pipeline object from either a raw estimator or a saved dict artifact."""
    if isinstance(artifact, dict):
        for key in ["model", "pipeline", "estimator", "best_estimator", "best_estimator_", "clf", "classifier"]:
            if key in artifact:
                artifact = artifact[key]
                break
    if not (hasattr(artifact, "decision_function") or hasattr(artifact, "predict_proba")):
        where = f" from {artifact_path}" if artifact_path is not None else ""
        raise TypeError(f"Loaded model artifact{where} has neither decision_function nor predict_proba.")
    return artifact

def load_model_artifact(model_path):
    artifact = joblib.load(model_path)
    return unwrap_model_artifact(artifact, artifact_path=model_path)

def unwrap_pca_artifact(artifact, pca_path=None):
    """Stage 03 PCA artifacts may be saved either directly or inside a dict."""
    if isinstance(artifact, dict):
        if "pca" in artifact:
            artifact = artifact["pca"]
        else:
            available = ", ".join(map(str, artifact.keys()))
            raise KeyError(f"PCA artifact dict does not contain key 'pca'. Available keys: {available}. Path: {pca_path}")
    if not hasattr(artifact, "transform"):
        raise TypeError(f"PCA artifact has no transform() method: {pca_path}")
    return artifact

def load_pca_artifact(dataset_name, patch_set, hog_variant):
    key = (dataset_name, patch_set, hog_variant)
    if key in PCA_ARTIFACT_CACHE:
        return PCA_ARTIFACT_CACHE[key]
    pca_path = DATASET_CONFIGS[dataset_name]["pca_artifacts_dir"] / f"{patch_set}__{hog_variant}_pca.joblib"
    if not pca_path.exists():
        raise FileNotFoundError(f"Missing PCA artifact: {pca_path}")
    artifact = joblib.load(pca_path)
    pca = unwrap_pca_artifact(artifact, pca_path=pca_path)
    PCA_ARTIFACT_CACHE[key] = pca
    return pca

def extract_hog_features(patch_bgr, pixels_per_cell):
    # cv2.imread returns BGR. BGR→GRAY here is equivalent to Stage 08's RGB→GRAY after RGB loading.
    # Keep uint8 grayscale input to match Stage 08 / Stage 03 HOG extraction behavior.
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    return hog(
        gray, orientations=9, pixels_per_cell=(pixels_per_cell, pixels_per_cell),
        cells_per_block=(2, 2), block_norm="L2-Hys", transform_sqrt=True,
        feature_vector=True, channel_axis=None,
    ).astype(np.float32)

def normalized_hist(values, bins, value_range):
    hist, _ = np.histogram(values, bins=bins, range=value_range)
    hist = hist.astype(np.float32)
    total = hist.sum()
    if total > 0:
        hist /= total
    return hist

def extract_hsv_features(patch_bgr):
    # cv2.imread returns BGR; direct BGR→HSV is intentionally used and matches RGB→HSV after RGB conversion.
    hsv = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)
    h = normalized_hist(hsv[:, :, 0].ravel(), 16, (0, 180))
    s = normalized_hist(hsv[:, :, 1].ravel(), 8, (0, 256))
    v = normalized_hist(hsv[:, :, 2].ravel(), 8, (0, 256))
    return np.concatenate([h, s, v]).astype(np.float32)

def extract_lbp_features(patch_bgr):
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    n_bins = 10  # P + 2 for P=8 uniform LBP.
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1))
    hist = hist.astype(np.float32)
    total = hist.sum()
    if total > 0:
        hist /= total
    return hist

def extract_feature_matrix_from_patches(patches_bgr, dataset_name, patch_set, feature_set):
    """Extract a feature matrix for one batch of patches.

    PCA is applied in batch after raw HOG extraction. No cross-model or disk cache is used.
    """
    if not patches_bgr:
        return np.empty((0, 0), dtype=np.float32)

    matrices = []

    if uses_hog(feature_set):
        hog_variant, ppc = parse_hog_variant(feature_set)
        if hog_variant is None:
            raise ValueError(f"Unsupported HOG variant in feature_set: {feature_set}")
        raw_hog = np.vstack([extract_hog_features(patch, ppc) for patch in patches_bgr]).astype(np.float32)
        if is_pca_feature_set(feature_set):
            pca = load_pca_artifact(dataset_name, patch_set, hog_variant)
            hog_matrix = pca.transform(raw_hog).astype(np.float32)
        else:
            hog_matrix = raw_hog
        matrices.append(hog_matrix)

    if uses_hsv(feature_set):
        matrices.append(np.vstack([extract_hsv_features(patch) for patch in patches_bgr]).astype(np.float32))

    if uses_lbp(feature_set):
        matrices.append(np.vstack([extract_lbp_features(patch) for patch in patches_bgr]).astype(np.float32))

    if not matrices:
        raise ValueError(f"Unsupported feature_set: {feature_set}")

    return np.hstack(matrices).astype(np.float32)

def extract_feature_vector_from_patch(patch_bgr, dataset_name, patch_set, feature_set):
    """Single-patch wrapper retained for debugging/sanity checks."""
    return extract_feature_matrix_from_patches([patch_bgr], dataset_name, patch_set, feature_set)[0]


## 11. Detection Yardımcıları

Bu bölümde sliding-window aday üretimi, model skorlaması ve weighted-center post-process işlemleri tanımlanır.


In [ ]:
def generate_positions(length, patch_size, stride):
    if length < patch_size:
        return []
    positions = list(range(0, max(1, length - patch_size + 1), stride))
    last = length - patch_size
    if not positions or positions[-1] != last:
        positions.append(last)
    return sorted(set(int(p) for p in positions))

def generate_sliding_windows(image_width, image_height, patch_size, stride_divisor):
    # Stage 08 compatibility: keep a minimum stride of 4 px.
    stride = max(4, int(patch_size // stride_divisor))
    xs = generate_positions(image_width, patch_size, stride)
    ys = generate_positions(image_height, patch_size, stride)
    windows = []
    for y1 in ys:
        for x1 in xs:
            windows.append({
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x1 + patch_size),
                "y2": float(y1 + patch_size),
                "cx": float(x1 + patch_size / 2),
                "cy": float(y1 + patch_size / 2),
            })
    return windows, stride

def compute_iou(a, b):
    inter_x1, inter_y1 = max(a["x1"], b["x1"]), max(a["y1"], b["y1"])
    inter_x2, inter_y2 = min(a["x2"], b["x2"]), min(a["y2"], b["y2"])
    inter_w, inter_h = max(0.0, inter_x2 - inter_x1), max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    area_a = max(0.0, a["x2"] - a["x1"]) * max(0.0, a["y2"] - a["y1"])
    area_b = max(0.0, b["x2"] - b["x1"]) * max(0.0, b["y2"] - b["y1"])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def clip_box(box, image_width, image_height):
    box = dict(box)
    max_x1 = max(0.0, float(image_width) - 1.0)
    max_y1 = max(0.0, float(image_height) - 1.0)
    box["x1"] = max(0.0, min(max_x1, box["x1"]))
    box["y1"] = max(0.0, min(max_y1, box["y1"]))
    box["x2"] = max(0.0, min(float(image_width), box["x2"]))
    box["y2"] = max(0.0, min(float(image_height), box["y2"]))
    if box["x2"] < box["x1"]:
        box["x1"], box["x2"] = box["x2"], box["x1"]
    if box["y2"] < box["y1"]:
        box["y1"], box["y2"] = box["y2"], box["y1"]
    return box

def score_feature_matrix(model, X, threshold_column):
    if threshold_column == "predict_proba_score_threshold":
        if not hasattr(model, "predict_proba"):
            raise AttributeError("Model does not expose predict_proba required for predict_proba_score_threshold.")
        scores = model.predict_proba(X)[:, 1]
    else:
        if hasattr(model, "decision_function"):
            scores = model.decision_function(X)
        elif hasattr(model, "predict_proba"):
            scores = model.predict_proba(X)[:, 1]
        else:
            raise AttributeError("Model exposes neither decision_function nor predict_proba.")
    return np.asarray(scores).ravel()

def score_image_windows(model, image_bgr, dataset_name, patch_set, patch_size, feature_set, stride_divisor, threshold_column):
    h, w = image_bgr.shape[:2]
    windows, stride = generate_sliding_windows(w, h, patch_size, stride_divisor)

    scored = []
    batch_patches, batch_windows = [], []

    def flush_batch():
        nonlocal batch_patches, batch_windows, scored
        if not batch_patches:
            return
        X = extract_feature_matrix_from_patches(
            batch_patches,
            dataset_name=dataset_name,
            patch_set=patch_set,
            feature_set=feature_set,
        )
        scores = score_feature_matrix(model, X, threshold_column)
        scored.extend([{**win, "score": float(score)} for win, score in zip(batch_windows, scores)])
        batch_patches, batch_windows = [], []

    for win in windows:
        x1, y1, x2, y2 = map(int, [win["x1"], win["y1"], win["x2"], win["y2"]])
        patch = image_bgr[y1:y2, x1:x2]
        if patch.shape[:2] != (patch_size, patch_size):
            continue
        batch_patches.append(patch)
        batch_windows.append(win)

        if len(batch_patches) >= FEATURE_EXTRACTION_BATCH_SIZE:
            flush_batch()

    flush_batch()
    return scored, stride

def build_candidate_detections(scored_windows, threshold):
    candidates = [w for w in scored_windows if w["score"] >= threshold]
    candidates = sorted(candidates, key=lambda d: d["score"], reverse=True)
    return candidates[:MAX_CANDIDATES_PER_IMAGE]

def weighted_center_cluster(candidates, cluster_iou_threshold, weighted_box_size_factor, patch_size, threshold, image_width, image_height):
    remaining = sorted(candidates, key=lambda d: d["score"], reverse=True)
    final = []
    eps = 1e-6
    while remaining:
        seed = remaining[0]
        cluster, keep = [seed], []
        for det in remaining[1:]:
            if compute_iou(seed, det) >= cluster_iou_threshold:
                cluster.append(det)
            else:
                keep.append(det)

        scores = np.array([d["score"] for d in cluster], dtype=np.float64)
        weights = np.maximum(scores - float(threshold) + eps, eps)
        cx = float(np.sum(np.array([d["cx"] for d in cluster]) * weights) / np.sum(weights))
        cy = float(np.sum(np.array([d["cy"] for d in cluster]) * weights) / np.sum(weights))

        final_size = float(patch_size) * float(weighted_box_size_factor)
        box = {
            "x1": cx - final_size / 2,
            "y1": cy - final_size / 2,
            "x2": cx + final_size / 2,
            "y2": cy + final_size / 2,
            "cx": cx,
            "cy": cy,
            "score": float(np.max(scores)),
            "mean_score": float(np.mean(scores)),
            "cluster_size": int(len(cluster)),
        }
        final.append(clip_box(box, image_width, image_height))
        remaining = keep

    return sorted(final, key=lambda d: d["score"], reverse=True)

def run_detection_for_image(model, image_record, dataset_name, patch_set, patch_size, feature_set, threshold_column, threshold, stride_divisor, cluster_iou_threshold, weighted_box_size_factor):
    start_time = time.time()
    image_bgr = cv2.imread(str(image_record["image_path"]))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_record['image_path']}")

    h, w = image_bgr.shape[:2]
    scored_windows, stride_px = score_image_windows(
        model=model,
        image_bgr=image_bgr,
        dataset_name=dataset_name,
        patch_set=patch_set,
        patch_size=patch_size,
        feature_set=feature_set,
        stride_divisor=stride_divisor,
        threshold_column=threshold_column,
    )
    candidates = build_candidate_detections(scored_windows, threshold)
    detections = weighted_center_cluster(
        candidates=candidates,
        cluster_iou_threshold=cluster_iou_threshold,
        weighted_box_size_factor=weighted_box_size_factor,
        patch_size=patch_size,
        threshold=threshold,
        image_width=w,
        image_height=h,
    )

    return {
        "image_id": image_record["image_id"],
        "gt_boxes": image_record["gt_boxes"],
        "detections": detections,
        "scored_window_count": len(scored_windows),
        "window_count": len(scored_windows),  # backward-compatible alias for quick inspection
        "candidate_count": len(candidates),
        "final_detection_count": len(detections),
        "stride_px": stride_px,
        "runtime_seconds": time.time() - start_time,
    }


## 12. Metrik Yardımcıları

Bu bölümde detection, image-level presence ve AP/mAP metrikleri hesaplanır.


In [ ]:
def match_detections_to_gt(detections, gt_boxes, iou_threshold):
    detections = sorted(detections, key=lambda d: d.get("score", 0.0), reverse=True)
    matched_gt = set()
    tp, fp = 0, 0
    matched_ious = []
    for det in detections:
        best_iou, best_idx = 0.0, None
        for idx, gt in enumerate(gt_boxes):
            if idx in matched_gt:
                continue
            iou = compute_iou(det, gt)
            if iou > best_iou:
                best_iou, best_idx = iou, idx
        if best_idx is not None and best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_idx)
            matched_ious.append(best_iou)
        else:
            fp += 1
    return {"tp": tp, "fp": fp, "fn": len(gt_boxes) - len(matched_gt), "matched_ious": matched_ious}

def evaluate_detection_outputs(image_outputs, iou_thresholds=MATCHING_IOU_THRESHOLDS):
    total_images = len(image_outputs)
    total_window_count = sum(o.get("scored_window_count", o.get("window_count", 0)) for o in image_outputs)
    total_candidate_count = sum(o.get("candidate_count", 0) for o in image_outputs)
    total_final_detection_count = sum(o.get("final_detection_count", len(o.get("detections", []))) for o in image_outputs)
    total_runtime_seconds = sum(o.get("runtime_seconds", 0.0) for o in image_outputs)
    gt_count = sum(len(o.get("gt_boxes", [])) for o in image_outputs)

    base = {
        "image_count": total_images,
        "gt_count": int(gt_count),
        "prediction_count": int(total_final_detection_count),
        "total_window_count": int(total_window_count),
        "total_candidate_count": int(total_candidate_count),
        "total_final_detection_count": int(total_final_detection_count),
        "mean_candidates_per_image": safe_divide(total_candidate_count, total_images),
        "mean_final_detections_per_image": safe_divide(total_final_detection_count, total_images),
        "total_runtime_seconds": float(total_runtime_seconds),
        "mean_runtime_seconds_per_image": safe_divide(total_runtime_seconds, total_images),
    }

    for iou_threshold in iou_thresholds:
        key = metric_key(iou_threshold)
        tp_total = fp_total = fn_total = 0
        matched_ious_all = []

        for output in image_outputs:
            match = match_detections_to_gt(output.get("detections", []), output.get("gt_boxes", []), iou_threshold)
            tp_total += match["tp"]
            fp_total += match["fp"]
            fn_total += match["fn"]
            matched_ious_all.extend(match["matched_ious"])

        precision = safe_divide(tp_total, tp_total + fp_total)
        recall = safe_divide(tp_total, tp_total + fn_total)
        f1 = safe_divide(2 * precision * recall, precision + recall)

        base.update({
            f"tp_iou_{key}": int(tp_total),
            f"fp_iou_{key}": int(fp_total),
            f"fn_iou_{key}": int(fn_total),
            f"detection_precision_iou_{key}": precision,
            f"detection_recall_iou_{key}": recall,
            f"detection_f1_iou_{key}": f1,
            f"false_positives_per_image_iou_{key}": safe_divide(fp_total, total_images),
            f"false_negatives_per_image_iou_{key}": safe_divide(fn_total, total_images),
            f"mean_matched_iou_iou_{key}": float(np.mean(matched_ious_all)) if matched_ious_all else 0.0,
        })
    return base

def compute_image_level_presence_metrics(image_outputs):
    tp = tn = fp = fn = 0
    for output in image_outputs:
        gt_has = len(output.get("gt_boxes", [])) > 0
        pred_has = len(output.get("detections", [])) > 0
        if gt_has and pred_has:
            tp += 1
        elif (not gt_has) and (not pred_has):
            tn += 1
        elif (not gt_has) and pred_has:
            fp += 1
        elif gt_has and (not pred_has):
            fn += 1

    total = tp + tn + fp + fn
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    f1 = safe_divide(2 * precision * recall, precision + recall)
    return {
        "image_tp": int(tp),
        "image_tn": int(tn),
        "image_fp": int(fp),
        "image_fn": int(fn),
        "image_accuracy": safe_divide(tp + tn, total),
        "image_precision": precision,
        "image_recall": recall,
        "image_specificity": specificity,
        "image_f1": f1,
        "false_positive_image_rate": safe_divide(fp, fp + tn),
        "missed_positive_image_rate": safe_divide(fn, fn + tp),
    }

def compute_ap_for_iou(detections_df, image_records_df, iou_threshold):
    # Build GT dictionary by image_id.
    gt_by_image = {str(row["image_id"]): list(row["gt_boxes"]) for _, row in image_records_df.iterrows()}
    total_gt = sum(len(v) for v in gt_by_image.values())
    if total_gt == 0:
        return 0.0

    if detections_df.empty:
        return 0.0

    detections = detections_df.sort_values("score", ascending=False).to_dict("records")
    matched = {image_id: set() for image_id in gt_by_image}
    tp_flags = []
    fp_flags = []

    for det in detections:
        image_id = str(det["image_id"])
        gt_boxes = gt_by_image.get(image_id, [])
        det_box = {"x1": det["x1"], "y1": det["y1"], "x2": det["x2"], "y2": det["y2"]}
        best_iou, best_idx = 0.0, None
        for idx, gt in enumerate(gt_boxes):
            if idx in matched.setdefault(image_id, set()):
                continue
            iou = compute_iou(det_box, gt)
            if iou > best_iou:
                best_iou, best_idx = iou, idx
        if best_idx is not None and best_iou >= iou_threshold:
            matched[image_id].add(best_idx)
            tp_flags.append(1.0)
            fp_flags.append(0.0)
        else:
            tp_flags.append(0.0)
            fp_flags.append(1.0)

    tp_cum = np.cumsum(tp_flags)
    fp_cum = np.cumsum(fp_flags)
    recalls = tp_cum / max(1, total_gt)
    precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)

    # COCO-style 101-point interpolation.
    recall_grid = np.linspace(0.0, 1.0, 101)
    interpolated = []
    for r in recall_grid:
        mask = recalls >= r
        interpolated.append(float(np.max(precisions[mask])) if np.any(mask) else 0.0)
    return float(np.mean(interpolated))

def compute_ap_map_for_group(final_detections_df, image_records_df):
    rows = []
    if final_detections_df.empty:
        base = {"AP@0.20": 0.0, "AP@0.30": 0.0, "AP@0.50": 0.0, "mAP50": 0.0, "mAP50-95": 0.0}
        return base

    ap_values = {}
    for iou in AP_REPORT_IOU_THRESHOLDS:
        ap_values[f"AP@{iou:.2f}"] = compute_ap_for_iou(final_detections_df, image_records_df, iou)
    map_values = [compute_ap_for_iou(final_detections_df, image_records_df, iou) for iou in MAP_50_95_THRESHOLDS]
    ap_values["mAP50"] = ap_values["AP@0.50"]
    ap_values["mAP50-95"] = float(np.mean(map_values)) if map_values else 0.0
    return ap_values

def sort_results(df, primary_metric=DEFAULT_PRIMARY_METRIC, iou_key="0_20"):
    if df.empty:
        return df.copy()
    fp_col = f"false_positives_per_image_iou_{iou_key}"
    recall_col = f"detection_recall_iou_{iou_key}"
    sort_cols = [primary_metric, fp_col, recall_col, STRICT_COMPARISON_METRIC, "mean_final_detections_per_image"]
    ascending = [False, True, False, False, True]
    existing = [(c, a) for c, a in zip(sort_cols, ascending) if c in df.columns]
    return df.sort_values([c for c, _ in existing], ascending=[a for _, a in existing]).reset_index(drop=True) if existing else df.copy()


## 13. Evaluation Runner

Bu bölümde sabit bir modeli seçilen test kayıtları üzerinde çalıştıran değerlendirme akışı tanımlanır.


In [ ]:
def make_partial_paths(eval_mode, dataset_name, model_id):
    safe_model = str(model_id)
    prefix = f"{eval_mode}__{dataset_name}__{safe_model}"
    return {
        "summary": PARTIAL_DIR / f"{prefix}__detection_summary.csv",
        "image_level": PARTIAL_DIR / f"{prefix}__image_level_presence.csv",
        "per_image": PARTIAL_DIR / f"{prefix}__per_image_records.csv",
        "detections": PARTIAL_DIR / f"{prefix}__final_detections.csv",
        "ap_map": PARTIAL_DIR / f"{prefix}__ap_map.csv",
    }

def outputs_to_per_image_df(image_outputs, eval_mode, dataset_name, model_row):
    rows = []
    for output in image_outputs:
        rows.append({
            "eval_mode": eval_mode,
            "dataset_name": dataset_name,
            "algorithm_name": model_row["algorithm_name"],
            "model_id": model_row["model_id"],
            "model_file_name": model_row["model_file_name"],
            "image_id": output["image_id"],
            "gt_count": len(output.get("gt_boxes", [])),
            "prediction_count": len(output.get("detections", [])),
            "scored_window_count": output.get("scored_window_count", 0),
            "candidate_count": output.get("candidate_count", 0),
            "final_detection_count": output.get("final_detection_count", len(output.get("detections", []))),
            "stride_px": output.get("stride_px", np.nan),
            "runtime_seconds": output.get("runtime_seconds", 0.0),
        })
    return pd.DataFrame(rows)

def outputs_to_final_detections_df(image_outputs, eval_mode, dataset_name, model_row):
    rows = []
    for output in image_outputs:
        image_id = output["image_id"]
        for det_index, det in enumerate(output.get("detections", []), start=1):
            rows.append({
                "eval_mode": eval_mode,
                "dataset_name": dataset_name,
                "algorithm_name": model_row["algorithm_name"],
                "model_id": model_row["model_id"],
                "model_file_name": model_row["model_file_name"],
                "image_id": image_id,
                "detection_index": det_index,
                "x1": float(det["x1"]),
                "y1": float(det["y1"]),
                "x2": float(det["x2"]),
                "y2": float(det["y2"]),
                "score": float(det.get("score", 0.0)),
                "mean_score": float(det.get("mean_score", det.get("score", 0.0))),
                "cluster_size": int(det.get("cluster_size", 1)),
            })
    return pd.DataFrame(rows)

def evaluate_fixed_model_on_records(eval_mode, dataset_name, model_row, image_records):
    paths = make_partial_paths(eval_mode, dataset_name, model_row["model_id"])
    if paths["summary"].exists() and not OVERWRITE_PARTIAL_RESULTS:
        print(f"Loading existing partial result: {paths['summary']}")
        summary_df = pd.read_csv(paths["summary"])
        image_level_df = pd.read_csv(paths["image_level"]) if paths["image_level"].exists() else pd.DataFrame()
        per_image_df = pd.read_csv(paths["per_image"]) if paths["per_image"].exists() else pd.DataFrame()
        detections_df = pd.read_csv(paths["detections"]) if paths["detections"].exists() else pd.DataFrame()
        ap_map_df = pd.read_csv(paths["ap_map"]) if paths["ap_map"].exists() else pd.DataFrame()
        return summary_df, image_level_df, per_image_df, detections_df, ap_map_df

    model_path = Path(model_row["model_path_resolved"])
    if not model_path.exists():
        raise FileNotFoundError(f"Missing model artifact: {model_path}")

    print(f"\n[{eval_mode} | {dataset_name}] {model_row['model_file_name']} | images={len(image_records)}")
    model = load_model_artifact(model_path)
    image_outputs = []
    start = time.time()

    for idx, (_, image_record) in enumerate(image_records.iterrows(), start=1):
        try:
            image_outputs.append(run_detection_for_image(
                model=model,
                image_record=image_record,
                dataset_name=dataset_name,
                patch_set=model_row["patch_set"],
                patch_size=int(model_row["patch_size"]),
                feature_set=model_row["feature_set"],
                threshold_column=model_row["threshold_column"],
                threshold=float(model_row["threshold_value"]),
                stride_divisor=int(model_row["stride_divisor"]),
                cluster_iou_threshold=float(model_row["cluster_iou_threshold"]),
                weighted_box_size_factor=float(model_row["weighted_box_size_factor"]),
            ))
        except Exception as exc:
            warnings.warn(f"Image failed: eval_mode={eval_mode}, dataset={dataset_name}, model={model_row['model_id']}, image={image_record.get('image_id')}: {exc}")
            image_outputs.append({
                "image_id": image_record.get("image_id"),
                "gt_boxes": image_record.get("gt_boxes", []),
                "detections": [],
                "scored_window_count": 0,
                "candidate_count": 0,
                "final_detection_count": 0,
                "stride_px": np.nan,
                "runtime_seconds": 0.0,
                "error_message": str(exc),
            })

        if idx % 50 == 0 or idx == len(image_records):
            elapsed = time.time() - start
            print(f"  processed {idx}/{len(image_records)} images | elapsed={elapsed/60:.1f} min")

    summary = {
        "eval_mode": eval_mode,
        "dataset_name": dataset_name,
        "algorithm_name": model_row["algorithm_name"],
        "model_id": model_row["model_id"],
        "model_file_name": model_row["model_file_name"],
        "model_path": relative_path(model_row["model_path_resolved"]),
        "patch_set": model_row["patch_set"],
        "patch_size": int(model_row["patch_size"]),
        "feature_set": model_row["feature_set"],
        "threshold_column": model_row["threshold_column"],
        "threshold_value": float(model_row["threshold_value"]),
        "stride_divisor": int(model_row["stride_divisor"]),
        "cluster_iou_threshold": float(model_row["cluster_iou_threshold"]),
        "weighted_box_size_factor": float(model_row["weighted_box_size_factor"]),
        "postprocess_method": POSTPROCESS_METHOD,
        "status": "OK",
    }
    summary.update(evaluate_detection_outputs(image_outputs))

    image_level = {
        "eval_mode": eval_mode,
        "dataset_name": dataset_name,
        "algorithm_name": model_row["algorithm_name"],
        "model_id": model_row["model_id"],
        "model_file_name": model_row["model_file_name"],
    }
    image_level.update(compute_image_level_presence_metrics(image_outputs))

    per_image_df = outputs_to_per_image_df(image_outputs, eval_mode, dataset_name, model_row)
    detections_df = outputs_to_final_detections_df(image_outputs, eval_mode, dataset_name, model_row)

    if COMPUTE_AP_MAP:
        ap_values = compute_ap_map_for_group(detections_df, image_records)
    else:
        ap_values = {"AP@0.20": np.nan, "AP@0.30": np.nan, "AP@0.50": np.nan, "mAP50": np.nan, "mAP50-95": np.nan}
    ap_map = {
        "eval_mode": eval_mode,
        "dataset_name": dataset_name,
        "algorithm_name": model_row["algorithm_name"],
        "model_id": model_row["model_id"],
        "model_file_name": model_row["model_file_name"],
    }
    ap_map.update(ap_values)

    summary_df = pd.DataFrame([summary])
    image_level_df = pd.DataFrame([image_level])
    ap_map_df = pd.DataFrame([ap_map])

    save_csv(summary_df, paths["summary"], f"Partial detection summary for {eval_mode} / {dataset_name} / {model_row['model_id']}")
    save_csv(image_level_df, paths["image_level"], f"Partial image-level presence results for {eval_mode} / {dataset_name} / {model_row['model_id']}")
    save_csv(per_image_df, paths["per_image"], f"Partial per-image detection records for {eval_mode} / {dataset_name} / {model_row['model_id']}")
    save_csv(detections_df, paths["detections"], f"Partial final detections for {eval_mode} / {dataset_name} / {model_row['model_id']}")
    save_csv(ap_map_df, paths["ap_map"], f"Partial AP/mAP results for {eval_mode} / {dataset_name} / {model_row['model_id']}")

    del model
    return summary_df, image_level_df, per_image_df, detections_df, ap_map_df

def run_evaluation_mode(eval_mode, dataset_name, records):
    model_rows = STAGE10_FIXED_PARAMS[STAGE10_FIXED_PARAMS["dataset_name"] == dataset_name].copy()
    outputs = []
    for _, model_row in model_rows.iterrows():
        try:
            outputs.append(evaluate_fixed_model_on_records(eval_mode, dataset_name, model_row, records))
        except Exception as exc:
            error_row = {
                "eval_mode": eval_mode,
                "dataset_name": dataset_name,
                "algorithm_name": model_row.get("algorithm_name"),
                "model_id": model_row.get("model_id"),
                "model_file_name": model_row.get("model_file_name"),
                "status": "FAILED",
                "error_message": str(exc),
                "traceback": traceback.format_exc(),
            }
            path = make_partial_paths(eval_mode, dataset_name, model_row["model_id"])["summary"]
            save_csv(pd.DataFrame([error_row]), path, f"Failed partial result for {eval_mode} / {dataset_name} / {model_row['model_id']}")
            warnings.warn(f"Model evaluation failed: {eval_mode} / {dataset_name} / {model_row['model_id']}: {exc}")
    return outputs


## 14. Evaluation Blok Yardımcıları

Bu bölümde seçilen test modu ve veri seti için Stage 10 değerlendirme bloğu hazırlanır.


In [ ]:
all_mode_outputs = []

def get_eval_records(eval_mode, dataset_name):
    matches = [
        mode for mode in eval_modes
        if mode["eval_mode"] == eval_mode and mode["dataset_name"] == dataset_name
    ]
    if not matches:
        warnings.warn(
            f"Evaluation block not found or disabled: eval_mode={eval_mode}, dataset_name={dataset_name}. "
            "This can happen if the related RUN_* flag is False."
        )
        return None
    return matches[0]["records"]

def run_stage10_block(eval_mode, dataset_name):
    records = get_eval_records(eval_mode, dataset_name)
    if records is None:
        return []

    print("=" * 100)
    print(f"Starting Stage 10 block: eval_mode={eval_mode} | dataset={dataset_name} | images={len(records)}")
    print("=" * 100)

    start_time = time.time()
    outputs = run_evaluation_mode(eval_mode, dataset_name, records)
    elapsed_minutes = (time.time() - start_time) / 60.0

    all_mode_outputs.extend(outputs)

    print("=" * 100)
    print(
        f"Completed Stage 10 block: eval_mode={eval_mode} | dataset={dataset_name} | "
        f"model_evaluations={len(outputs)} | elapsed_minutes={elapsed_minutes:.2f}"
    )
    print("=" * 100)

    return outputs


## 15. Natural 700 - Dataset1

Bu bölüm Dataset1 üzerinde doğal 700 test alt kümesini çalıştırır.


In [ ]:
if RUN_NATURAL_700:
    natural_700_dataset1_outputs = run_stage10_block("natural_700", "dataset1_varroa")
else:
    natural_700_dataset1_outputs = []
    print("Skipped natural_700 / dataset1_varroa because RUN_NATURAL_700=False.")


## 16. Natural 700 - Dataset2

Bu bölüm Dataset2 üzerinde doğal 700 test alt kümesini çalıştırır.


In [ ]:
if RUN_NATURAL_700:
    natural_700_dataset2_outputs = run_stage10_block("natural_700", "dataset2_honeybee")
else:
    natural_700_dataset2_outputs = []
    print("Skipped natural_700 / dataset2_honeybee because RUN_NATURAL_700=False.")


## 17. Controlled 550 - Dataset1

Bu bölüm Dataset1 üzerinde kontrollü 550 test alt kümesini çalıştırır.


In [ ]:
if RUN_CONTROLLED_550:
    controlled_550_dataset1_outputs = run_stage10_block("controlled_550", "dataset1_varroa")
else:
    controlled_550_dataset1_outputs = []
    print("Skipped controlled_550 / dataset1_varroa because RUN_CONTROLLED_550=False.")


## 18. Controlled 550 - Dataset2

Bu bölüm Dataset2 üzerinde kontrollü 550 test alt kümesini çalıştırır.


In [ ]:
if RUN_CONTROLLED_550:
    controlled_550_dataset2_outputs = run_stage10_block("controlled_550", "dataset2_honeybee")
else:
    controlled_550_dataset2_outputs = []
    print("Skipped controlled_550 / dataset2_honeybee because RUN_CONTROLLED_550=False.")


## 19. Full Test - Dataset1

Bu bölüm Dataset1 tam test splitini çalıştırır.


In [ ]:
if RUN_FULL_TEST:
    full_test_dataset1_outputs = run_stage10_block("full_test", "dataset1_varroa")
else:
    full_test_dataset1_outputs = []
    print("Skipped full_test / dataset1_varroa because RUN_FULL_TEST=False.")


## 20. Full Test - Dataset2

Bu bölüm Dataset2 tam test splitini çalıştırır.


In [ ]:
if RUN_FULL_TEST:
    full_test_dataset2_outputs = run_stage10_block("full_test", "dataset2_honeybee")
else:
    full_test_dataset2_outputs = []
    print("Skipped full_test / dataset2_honeybee because RUN_FULL_TEST=False.")

print("Completed all requested Stage 10 evaluation cells that were executed.")


## 21. Sonuç Birleştirme ve Sıralama

Bu bölüm partial model çıktılarını birleştirir ve Stage 10 final tablolarını üretir.


In [ ]:
def load_partial_csvs(suffix):
    frames = []
    for path in sorted(PARTIAL_DIR.glob(f"*__{suffix}.csv")):
        try:
            frames.append(pd.read_csv(path))
        except Exception as exc:
            warnings.warn(f"Could not read partial file {path}: {exc}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

stage10_detection_results_df = load_partial_csvs("detection_summary")
stage10_image_level_df = load_partial_csvs("image_level_presence")
stage10_per_image_df = load_partial_csvs("per_image_records")
stage10_final_detections_df = load_partial_csvs("final_detections")
stage10_ap_map_df = load_partial_csvs("ap_map")

save_csv(stage10_detection_results_df, TABLES_DIR / "stage10_detection_results.csv", "Combined Stage 10 detection-level results")
save_csv(stage10_image_level_df, TABLES_DIR / "stage10_image_level_presence_results.csv", "Combined Stage 10 image-level Varroa presence/absence results")
save_csv(stage10_per_image_df, TABLES_DIR / "stage10_per_image_detection_records.csv", "Combined Stage 10 per-image detection records")
save_csv(stage10_final_detections_df, TABLES_DIR / "stage10_final_detections.csv", "Combined Stage 10 final detections")
save_csv(stage10_ap_map_df, TABLES_DIR / "stage10_ap_map_results.csv", "Combined Stage 10 AP/mAP results")

ranked_iou_0_20 = sort_results(stage10_detection_results_df, primary_metric="detection_f1_iou_0_20", iou_key="0_20")
ranked_iou_0_30 = sort_results(stage10_detection_results_df, primary_metric="detection_f1_iou_0_30", iou_key="0_30")

save_csv(ranked_iou_0_20, TABLES_DIR / "stage10_ranked_detection_results_by_iou_0_20.csv", "Stage 10 ranked detection results by F1@0.20")
save_csv(ranked_iou_0_30, TABLES_DIR / "stage10_ranked_detection_results_by_iou_0_30.csv", "Stage 10 ranked detection results by F1@0.30")

# Compact reporting table for quick notebook display.
compact_cols = [
    "eval_mode", "dataset_name", "algorithm_name", "model_id", "model_file_name",
    "image_count", "gt_count", "prediction_count",
    "detection_f1_iou_0_20", "detection_precision_iou_0_20", "detection_recall_iou_0_20",
    "tp_iou_0_20", "fp_iou_0_20", "fn_iou_0_20", "false_positives_per_image_iou_0_20",
    "detection_f1_iou_0_30", "detection_precision_iou_0_30", "detection_recall_iou_0_30",
    "tp_iou_0_30", "fp_iou_0_30", "fn_iou_0_30", "false_positives_per_image_iou_0_30",
    "mean_final_detections_per_image", "total_runtime_seconds", "status",
]
compact_cols = [c for c in compact_cols if c in stage10_detection_results_df.columns]
stage10_compact_results_df = ranked_iou_0_20[compact_cols].copy() if compact_cols else ranked_iou_0_20.copy()
save_csv(stage10_compact_results_df, TABLES_DIR / "stage10_compact_detection_results.csv", "Compact Stage 10 detection results")

display(stage10_compact_results_df.head(30))
display(stage10_image_level_df.head(30))
display(stage10_ap_map_df.head(30))


## 22. Final Durum

Bu bölüm notebook durumunu ve ana çıktı listesini kaydeder.


In [ ]:
completed_model_evals = 0 if stage10_detection_results_df.empty else int((stage10_detection_results_df.get("status", pd.Series(dtype=str)) == "OK").sum())
failed_model_evals = 0 if stage10_detection_results_df.empty else int((stage10_detection_results_df.get("status", pd.Series(dtype=str)) == "FAILED").sum())
expected_model_evals = sum(len(STAGE10_FIXED_PARAMS[STAGE10_FIXED_PARAMS["dataset_name"] == mode["dataset_name"]]) for mode in eval_modes)

notebook_status_df = pd.DataFrame([{
    "stage_name": STAGE_NAME,
    "notebook_name": NOTEBOOK_NAME,
    "status": "READY_FOR_REVIEW" if failed_model_evals == 0 else "COMPLETED_WITH_FAILURES",
    "started_at": NOTEBOOK_START_TIME.isoformat(timespec="seconds"),
    "finished_at": datetime.now().isoformat(timespec="seconds"),
    "run_natural_700": RUN_NATURAL_700,
    "run_controlled_550": RUN_CONTROLLED_550,
    "run_full_test": RUN_FULL_TEST,
    "compute_ap_map": COMPUTE_AP_MAP,
    "expected_model_evaluations": expected_model_evals,
    "completed_model_evaluations": completed_model_evals,
    "failed_model_evaluations": failed_model_evals,
    "finalist_selection_performed": False,
    "notes": "Stage 10 fixed-parameter internal test. No tuning or finalist auto-selection performed.",
}])
save_csv(notebook_status_df, TABLES_DIR / "notebook_status.csv", "Stage 10 notebook status")

output_summary_df = pd.DataFrame([
    {"output_name": "stage10_detection_results.csv", "path": relative_path(TABLES_DIR / "stage10_detection_results.csv")},
    {"output_name": "stage10_ranked_detection_results_by_iou_0_20.csv", "path": relative_path(TABLES_DIR / "stage10_ranked_detection_results_by_iou_0_20.csv")},
    {"output_name": "stage10_ranked_detection_results_by_iou_0_30.csv", "path": relative_path(TABLES_DIR / "stage10_ranked_detection_results_by_iou_0_30.csv")},
    {"output_name": "stage10_image_level_presence_results.csv", "path": relative_path(TABLES_DIR / "stage10_image_level_presence_results.csv")},
    {"output_name": "stage10_ap_map_results.csv", "path": relative_path(TABLES_DIR / "stage10_ap_map_results.csv")},
    {"output_name": "stage10_per_image_detection_records.csv", "path": relative_path(TABLES_DIR / "stage10_per_image_detection_records.csv")},
    {"output_name": "stage10_final_detections.csv", "path": relative_path(TABLES_DIR / "stage10_final_detections.csv")},
])
save_csv(output_summary_df, TABLES_DIR / "stage10_output_summary.csv", "Stage 10 output summary")

display(notebook_status_df)
display(output_summary_df)
